In [2]:
# Célula de Setup
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

In [3]:
# Criando a sessão do Spark (o ponto de entrada)
spark = SparkSession.builder \
    .appName("Lazy") \
    .master("local[*]") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

# Ajustando o nível de log para não poluir a tela
spark.sparkContext.setLogLevel("ERROR")

print("Spark iniciado com sucesso!")

Spark iniciado com sucesso!


#### Definindo as Regras (Transformações Lazy)

In [6]:
print("--- Definindo a Lógica de Processamento ---")
start_time = time.time()

# 1. Ler os dados (Apontamos para a pasta, não para um arquivo específico)
df_vendas = spark.read.parquet("data/vendas_parquet")
df_usuarios = spark.read.parquet("data/usuarios_parquet")

# 2. Aplicar regras de negócio complexas
# Note: Estamos apenas encadeando instruções.
df_relatorio = df_vendas.filter(F.col("valor") > 500) \
    .withColumn("valor_dolar", F.col("valor") / 5.0) \
    .withColumn("status", F.when(F.col("valor") > 2000, "Premium").otherwise("Padrão")) \
    .withColumn("ano", F.year(F.from_unixtime("timestamp"))) \
    .select("id_venda", "produto", "valor", "valor_dolar", "status", "ano")

end_time = time.time()

print(f"Tempo de execução: {end_time - start_time:.4f} segundos")
print("Status: O Spark processou os dados? Não!")

--- Definindo a Lógica de Processamento ---
Tempo de execução: 1.5282 segundos
Status: O Spark processou os dados? Não!


#### Visualizando o Plano de Execução

In [7]:
print("--- O Plano do Arquiteto (Logical & Physical Plan) ---")

# O explain mostra o que o Spark PRETENDE fazer
df_relatorio.explain()

--- O Plano do Arquiteto (Logical & Physical Plan) ---
== Physical Plan ==
*(1) Project [id_venda#98, produto#100, valor#101, (valor#101 / 5.0) AS valor_dolar#117, CASE WHEN (valor#101 > 2000.0) THEN Premium ELSE Padrão END AS status#124, year(cast(from_unixtime(cast(timestamp#102 as bigint), yyyy-MM-dd HH:mm:ss, Some(Etc/UTC)) as date)) AS ano#132]
+- *(1) Filter (isnotnull(valor#101) AND (valor#101 > 500.0))
   +- *(1) ColumnarToRow
      +- FileScan parquet [id_venda#98,produto#100,valor#101,timestamp#102] Batched: true, DataFilters: [isnotnull(valor#101), (valor#101 > 500.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/vendas_parquet], PartitionFilters: [], PushedFilters: [IsNotNull(valor), GreaterThan(valor,500.0)], ReadSchema: struct<id_venda:int,produto:string,valor:double,timestamp:int>




#### Executando de Verdade (Action)

In [8]:
print("--- ACORDANDO O SPARK (Action) ---")
start_time = time.time()

# O .count() obriga o Spark a percorrer todo o plano desenhado acima
total_linhas = df_relatorio.count()

end_time = time.time()

print(f"Total de Vendas no Relatório: {total_linhas}")
print(f"Tempo REAL de Processamento: {end_time - start_time:.4f} segundos")

--- ACORDANDO O SPARK (Action) ---
Total de Vendas no Relatório: 4518594
Tempo REAL de Processamento: 2.6556 segundos


#### Visualizando os Dados

In [9]:
print("--- Amostra dos Dados ---")
df_relatorio.show(5, truncate=False)

--- Amostra dos Dados ---
+--------+---------+-------+------------------+-------+----+
|id_venda|produto  |valor  |valor_dolar       |status |ano |
+--------+---------+-------+------------------+-------+----+
|1       |TV 4K    |3449.5 |689.9             |Premium|2023|
|2       |TV 4K    |3365.72|673.144           |Premium|2023|
|3       |TV 4K    |506.17 |101.23400000000001|Padrão |2023|
|4       |TV 4K    |2948.68|589.736           |Premium|2023|
|5       |Geladeira|3240.54|648.108           |Premium|2023|
+--------+---------+-------+------------------+-------+----+
only showing top 5 rows



# ⚡ Lazy vs. Eager: O Jogo da Espera

No Spark, existem dois tipos de comandos:

1.  **Transformações (Lazy/Preguiçosas):** O Spark apenas "anota o pedido" no plano de execução (DAG). Não processa nada. O retorno é um novo DataFrame.
    * *Exemplos:* `filter`, `select`, `withColumn`, `groupBy`, `join`, `orderBy`.
2.  **Ações (Eager/Ansiosas):** O Spark é obrigado a executar o plano para entregar um resultado (número, lista ou arquivo). O retorno é um valor ou dado, não um DataFrame.
    * *Exemplos:* `show`, `count`, `collect`, `take`, `write`.